In [1]:
import numpy as np
import numpy as np
import xarray as xr
import os
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import pandas as pd
#import pygmt
import sys
from scipy.ndimage import gaussian_filter
from datetime import datetime, timedelta
from scipy.interpolate import RegularGridInterpolator, RectBivariateSpline
import cartopy.feature as cfeature
import matplotlib.ticker as mticker
import netCDF4 as nc

import glob
from mudpy.forward import lowpass

In [46]:
station="NZL"
file=f"/Users/seansantellanes/Documents/infragravity_waves/PTHA_codes/{station}/pressure_res_long.h5"
ds=pd.read_hdf(file)
ds["datetime"] = pd.to_datetime(ds.iloc[:,1],unit="s").dt.tz_localize(None)
ds

,value,time,datetime
timestamp,,,
2023-09-22 00:00:00+00:00,0.070425,2023-09-22 00:00:00+00:00,2023-09-22 00:00:00
2023-09-22 00:00:15+00:00,0.071725,2023-09-22 00:00:15+00:00,2023-09-22 00:00:15
2023-09-22 00:00:30+00:00,0.071525,2023-09-22 00:00:30+00:00,2023-09-22 00:00:30
2023-09-22 00:00:45+00:00,0.071225,2023-09-22 00:00:45+00:00,2023-09-22 00:00:45
2023-09-22 00:01:00+00:00,0.071025,2023-09-22 00:01:00+00:00,2023-09-22 00:01:00
...,...,...,...
2024-12-16 23:58:45+00:00,0.009225,2024-12-16 23:58:45+00:00,2024-12-16 23:58:45
2024-12-16 23:59:00+00:00,0.009925,2024-12-16 23:59:00+00:00,2024-12-16 23:59:00
2024-12-16 23:59:15+00:00,0.009725,2024-12-16 23:59:15+00:00,2024-12-16 23:59:15


In [50]:
pd.options.mode.chained_assignment = None  # default='warn'

begin_time=datetime(2023,9,22)
end_time=datetime(2024,12,16)
station=station
file=f"/Users/seansantellanes/Documents/infragravity_waves/PTHA_codes/{station}/pressure_res_long.h5"
ds_era=pd.read_csv(f"ERA5_data/reanalysis-era5-single-levels-timeseries-{station}.csv")
ds_era["valid_time"] = pd.to_datetime(ds_era.iloc[:,0])
f=open(f"/Users/seansantellanes/Documents/infragravity_waves/water_level_codes/outputs/txt/{station}_meteotsunamis.txt","a")
df=pd.DataFrame()
while begin_time <= end_time:
    
    try:
        time_delta=begin_time+timedelta(days=4)  
        
        one_day=ds[ds["datetime"].between(begin_time,time_delta)]
        sf_pres_resid=one_day.iloc[:,0].to_numpy()
        one_day["IGW"] = lowpass(sf_pres_resid,fcorner=[1/600,1/32],fsample=1/15,order=4)
        igw_data = one_day["IGW"].to_numpy()
        one_day = one_day.set_index("datetime")
        ds_era_2=ds_era[ds_era["valid_time"].between(begin_time,time_delta)]
        shw=ds_era_2["msl"].to_numpy() #or msl
        shw=(shw-np.mean(shw))/np.std(shw)
        one_day = one_day.resample("1h").max()
        dart_wh=one_day.iloc[:-1,-1].to_numpy()
        std_dart=np.copy(dart_wh)
        dart_wh=(dart_wh-np.mean(dart_wh))/np.std(dart_wh)
        ccf=np.correlate(shw,dart_wh,mode="full")/(np.std(dart_wh)*np.std(shw)*len(shw))
        lags=np.arange(-len(shw),len(shw))
        max_ccf=np.max(abs(ccf))
        peak_to_trough=np.abs(igw_data.max()-igw_data.min())
        if max_ccf >= 0.75 and peak_to_trough >= 0.005:
            data={"begin_time":begin_time,"end_time":time_delta,"max_ccf":max_ccf,"peak_to_trough":peak_to_trough}
            df=pd.concat([df,pd.DataFrame([data])])
            print(f"{30*"-"}",file=f)
            print(begin_time,file=f)
            print(time_delta,file=f)
            print(max_ccf,file=f)
            print(peak_to_trough,file=f)
            print("Meteotsunami?",file=f)
            print(f"{30*"-"}",file=f)
        begin_time=time_delta
    except ValueError:
        time_delta=begin_time+timedelta(days=4)
        begin_time=time_delta
        continue
f.close()
print(df)
df.to_csv(f"/Users/seansantellanes/Documents/infragravity_waves/water_level_codes/outputs/csv/{station}_meteotsunami.csv",index=False)

Empty DataFrame
Columns: []
Index: []


In [51]:
new_data_try=pd.read_csv(f"/Users/seansantellanes/Documents/infragravity_waves/water_level_codes/outputs/csv/{station}_meteotsunami.csv")
ms_list=new_data_try.iloc[:,0].to_list()
ms_list

EmptyDataError: No columns to parse from file

In [11]:
data={"begin_time":begin_time,"end_time":time_delta,"max_ccf":max_ccf,"peak_to_trough":peak_to_trough}

In [12]:
data

{'begin_time': datetime.datetime(2022, 5, 3, 0, 0),
 'end_time': datetime.datetime(2022, 5, 3, 0, 0),
 'max_ccf': np.float64(0.47714436664064325),
 'peak_to_trough': np.float64(0.009724445182688309)}

In [13]:
df=pd.DataFrame()

In [17]:
df=pd.concat([df,pd.DataFrame([data])])
df

,begin_time,end_time,max_ccf,peak_to_trough
0,2022-05-03,2022-05-03,0.477144,0.009724
0,2022-05-03,2022-05-03,0.477144,0.009724


In [39]:
df=pd.read_hdf(f"/Users/seansantellanes/Documents/infragravity_waves/PTHA_codes/{station}/pressure_res_long.h5",parse_dates=['1'],index_col=['1'])
df["datetime"] = pd.to_datetime(df.iloc[:,1],unit="s")
df=df.set_index("datetime")
df

,0,1
datetime,,
2016-06-20 08:42:00,0.0416,1.466412e+09
2016-06-20 08:42:15,0.0240,1.466412e+09
2016-06-20 08:42:30,0.0203,1.466412e+09
2016-06-20 08:42:45,0.0178,1.466412e+09
2016-06-20 08:43:00,0.0153,1.466412e+09
...,...,...
2022-05-03 13:26:45,0.0384,1.651584e+09
2022-05-03 13:27:00,0.0403,1.651584e+09
2022-05-03 13:27:15,0.0411,1.651584e+09
